Extraction et traitement des données

Soit, télécharger les données, puis garder seulement les variables voulues, puis ajouter celles qu'on souhaite.

In [19]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)



In [20]:
DATA_DIR = Path("../data")

per100 = pd.read_csv(DATA_DIR / "Per_100_Poss.csv")
player_info = pd.read_csv(DATA_DIR / "Player_Season_Info.csv")
team_summary = pd.read_csv(DATA_DIR / "Team_Summaries.csv")
team_abbrev = pd.read_csv(DATA_DIR / "Team_Abbrev.csv")
all_star = pd.read_csv(DATA_DIR / "All-Star_Selections.csv")
end_season = pd.read_csv(DATA_DIR / "End_of_Season_Teams.csv")
team_p100 = pd.read_csv(DATA_DIR / "Team_Stats_Per_100_Poss.csv")
op_team_p100 =  pd.read_csv(DATA_DIR / "Opponent_Stats_Per_100_Poss.csv")

In [21]:
datasets = {
    "Per100": per100,
    "Player_Info": player_info,
    "Team_Summary": team_summary,
    "Team_Abbrev": team_abbrev,
    "All_Star": all_star,
    "End_Season": end_season
}

for name, df in datasets.items():
    print(name)
    print("Shape :", df.shape)
    print("Colonnes :", df.columns.tolist())

Per100
Shape : (27692, 34)
Colonnes : ['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs', 'mp', 'fg_per_100_poss', 'fga_per_100_poss', 'fg_percent', 'x3p_per_100_poss', 'x3pa_per_100_poss', 'x3p_percent', 'x2p_per_100_poss', 'x2pa_per_100_poss', 'x2p_percent', 'e_fg_percent', 'ft_per_100_poss', 'fta_per_100_poss', 'ft_percent', 'orb_per_100_poss', 'drb_per_100_poss', 'trb_per_100_poss', 'ast_per_100_poss', 'stl_per_100_poss', 'blk_per_100_poss', 'tov_per_100_poss', 'pf_per_100_poss', 'pts_per_100_poss', 'o_rtg', 'd_rtg']
Player_Info
Shape : (33339, 8)
Colonnes : ['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'experience']
Team_Summary
Shape : (1907, 10)
Colonnes : ['season', 'lg', 'team', 'abbreviation', 'playoffs', 'age', 'w', 'l', 'pw', 'pl']
Team_Abbrev
Shape : (1818, 5)
Colonnes : ['season', 'lg', 'team', 'abbreviation', 'playoffs']
All_Star
Shape : (2058, 6)
Colonnes : ['player', 'player_id', 'team', 'season', 'lg', 'replaced']
End_Season
Shape 

In [22]:
print("Per100 :", per100["season"].min(), "→", per100["season"].max())
print("Player info :", player_info["season"].min(), "→", player_info["season"].max())
print("Team summary :", team_summary["season"].min(), "→", team_summary["season"].max())

Per100 : 1974 → 2026
Player info : 1947 → 2026
Team summary : 1947 → 2026


Debut des per 100 en 1974, et selection des siasons avec seulement 82 matchs pour eviter des billets

In [23]:
numeric_cols = per100.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_cols

per100[numeric_cols] = per100[numeric_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [24]:
per100 = per100[per100["lg"] == "NBA"].copy()

Nettoyage des données

In [25]:
def clean_basic(df):
    df = df.copy()

    # espaces inutiles dans les noms
    df.columns = df.columns.str.strip()

    # suppression des lignes complètement vides
    df = df.dropna(how="all")

    # suppression des espaces autour des chaînes
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()

    # suppression des lignes 2TM, 3TM, etc.
    if "team" in df.columns:
        df = df[~df["team"].str.match(r"^\d+TM$", na=False)]

    return df

In [26]:
per100 = clean_basic(per100)
player_info = clean_basic(player_info)
team_summary = clean_basic(team_summary)
team_abbrev = clean_basic(team_abbrev)
all_star = clean_basic(all_star)
end_season = clean_basic(end_season)

C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\1781369532.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:
C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\1781369532.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

In [27]:
# Saisons où les équipes ont joué 82 matchs
team_summary["g"] = team_summary["l"] + team_summary["w"]

seasons_82 = (
    team_summary
    .loc[team_summary["g"] == 82, "season"]
    .unique()
)

print(f"Nombre de saisons à 82 matchs : {len(seasons_82)}")
print(seasons_82)

Nombre de saisons à 82 matchs : 55
[2026 2025 2024 2023 2022 2019 2018 2017 2016 2015 2014 2013 2011 2010
 2009 2008 2007 2006 2005 2004 2003 2002 2001 2000 1998 1997 1996 1995
 1994 1993 1992 1991 1990 1989 1988 1987 1986 1985 1984 1983 1982 1981
 1980 1979 1978 1977 1976 1975 1974 1973 1972 1971 1970 1969 1968]


In [28]:
team_summary = team_summary[
    team_summary["season"].isin(seasons_82)
].copy()

player_info = player_info[
    player_info["season"].isin(seasons_82)
].copy()

all_star = all_star[
    all_star["season"].isin(seasons_82)
].copy()

end_season = end_season[
    end_season["season"].isin(seasons_82)
].copy()

per100 = per100[per100["season"].isin(seasons_82)].copy()


In [29]:
datasets = {
    "team_p100": team_p100,
    "op_team_p100": op_team_p100
}

for name, df in datasets.items():

    # Années >= 1974
    df = df[df["season"] >= 1974].copy()

    # Supprimer les lignes agrégées des joueurs ayant joué
    # pour plusieurs équipes dans la même saison
    df = df[
        ~df["team"].astype(str).str.contains(r"TM$", na=False)
    ].copy()

    # Vérifier la clé
    key = [ "team", "season"]

    duplicates = (
        df
        .groupby(key)
        .size()
        .reset_index(name="n")
        .query("n > 1")
    )

    print(f"\n--- {name} ---")
    print(f"Nombre de lignes : {len(df):,}")
    print(f"Doublons sur {key} : {len(duplicates):,}")

    if len(duplicates) > 0:
        display(duplicates.head(20))
    else:
        print("La clé player_id + team + season est unique.")

    # Mettre à jour le DataFrame
    datasets[name] = df

team_p100 = datasets["team_p100"]
op_team_p100 = datasets["op_team_p100"]


--- team_p100 ---
Nombre de lignes : 1,462
Doublons sur ['team', 'season'] : 0
La clé player_id + team + season est unique.

--- op_team_p100 ---
Nombre de lignes : 1,462
Doublons sur ['team', 'season'] : 0
La clé player_id + team + season est unique.


Création des scores attribués (all-star et all-nba (defensive-team))

In [30]:
all_star_score = (
    all_star[["player_id", "season"]]
    .drop_duplicates()
    .assign(all_star_score=1)
)

all_star_score.head()

,player_id,season,all_star_score
0,barnesc01,2026,1
1,bookede01,2026,1
2,cunnica01,2026,1
3,durenja01,2026,1
4,edwaran01,2026,1


In [31]:
# Garder uniquement les sélections All-NBA
all_nba = end_season[
    end_season["type"] == "All-NBA"
].copy()

# Attribuer un score selon l'équipe All-NBA
all_nba["all_nba_score"] = all_nba["number_tm"].map({
    "1st": 3,
    "2nd": 2,
    "3rd": 1
})

# Garder uniquement ce dont on a besoin
all_nba_score = all_nba[
    ["player_id", "season", "all_nba_score"]
].copy()

# Vérification
display(all_nba_score.head(5))

,player_id,season,all_nba_score
10,jokicni01,2025,3
11,antetgi01,2025,3
12,tatumja01,2025,3
13,gilgesh01,2025,3
14,mitchdo01,2025,3


In [32]:
player_awards = pd.merge(
    all_star_score,
    all_nba_score,
    on=["player_id", "season"],
    how="outer"
).fillna(0)

display(player_awards.head(5))

,player_id,season,all_star_score,all_nba_score
0,abdulka01,1970,1.0,2.0
1,abdulka01,1971,1.0,3.0
2,abdulka01,1972,1.0,3.0
3,abdulka01,1973,1.0,3.0
4,abdulka01,1974,1.0,3.0


Il faut maintenant merge celui avec les poids qu'on attribue, personellement un lag de 1 an vaut un poid de 1, d il y a 2 ans vaut 0.75 et 3 ans 0.5

In [33]:
player_awards_team = player_awards.merge(
    player_info[["player_id", "season", "team"]],
    on=["player_id", "season"],
    how="left"
)

In [34]:
display(player_awards_team.head(5))

,player_id,season,all_star_score,all_nba_score,team
0,abdulka01,1970,1.0,2.0,MIL
1,abdulka01,1971,1.0,3.0,MIL
2,abdulka01,1972,1.0,3.0,MIL
3,abdulka01,1973,1.0,3.0,MIL
4,abdulka01,1974,1.0,3.0,MIL


In [35]:
team_awards = (
    player_awards_team
    .groupby(["team", "season"], as_index=False)
    .agg(
        all_star_score=("all_star_score", "sum"),
        all_nba_score=("all_nba_score", "sum")
    )
)

display(team_awards[team_awards['team']== 'MIL'].head(5))

,team,season,all_star_score,all_nba_score
535,MIL,1969,1.0,0.0
536,MIL,1970,2.0,2.0
537,MIL,1971,2.0,5.0
538,MIL,1972,2.0,3.0
539,MIL,1973,2.0,3.0


Creation de fonctions pour faire des sommes et ecart types ponderées

In [36]:
import numpy as np

def weighted_mean_std(row, cols, weights):

    values = row[cols].values.astype(float)

    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return pd.Series([np.nan, np.nan])

    values = values[mask]
    weights = np.array(weights)[mask]

    weighted_mean = np.average(
        values,
        weights=weights
    )

    weighted_var = np.average(
        (values - weighted_mean) ** 2,
        weights=weights
    )

    weighted_std = np.sqrt(weighted_var)

    return pd.Series([
        weighted_mean,
        weighted_std
    ])

weights = [1.0, 0.7, 0.4]

Création des variables precdente dans le temps

In [37]:
team_awards = team_awards.sort_values(["team", "season"])

team_awards["all_star_prev1"] = (
    team_awards.groupby("team")["all_star_score"].shift(1)
)

team_awards["all_star_prev2"] = (
    team_awards.groupby("team")["all_star_score"].shift(2)
)

team_awards["all_star_prev3"] = (
    team_awards.groupby("team")["all_star_score"].shift(3)
)

team_awards["all_nba_prev1"] = (
    team_awards.groupby("team")["all_nba_score"].shift(1)
)

team_awards["all_nba_prev2"] = (
    team_awards.groupby("team")["all_nba_score"].shift(2)
)

team_awards["all_nba_prev3"] = (
    team_awards.groupby("team")["all_nba_score"].shift(3)
)

In [38]:
award_cols = [
    "all_star_prev1",
    "all_star_prev2",
    "all_star_prev3",
    "all_nba_prev1",
    "all_nba_prev2",
    "all_nba_prev3"
]

team_awards[award_cols] = team_awards[award_cols].fillna(0)

In [39]:
# ALL-STAR
all_star_cols = [
    "all_star_prev1",
    "all_star_prev2",
    "all_star_prev3"
]

team_awards[["all_star_mean", "all_star_std"]] = (
    team_awards[all_star_cols].apply(
        lambda row: weighted_mean_std(
            row,
            all_star_cols,
            weights
        ),
        axis=1
    )
)


# ALL-NBA
all_nba_cols = [
    "all_nba_prev1",
    "all_nba_prev2",
    "all_nba_prev3"
]

team_awards[["all_nba_mean", "all_nba_std"]] = (
    team_awards[all_nba_cols].apply(
        lambda row: weighted_mean_std(
            row,
            all_nba_cols,
            weights
        ),
        axis=1
    )
)

In [40]:
display(
    team_awards[
        (team_awards["season"] == 2022)
    ]
)

,team,season,all_star_score,all_nba_score,all_star_prev1,all_star_prev2,all_star_prev3,all_nba_prev1,all_nba_prev2,all_nba_prev3,all_star_mean,all_star_std,all_nba_mean,all_nba_std
34,ATL,2022,1.0,1.0,1.0,2.0,4.0,0.0,0.0,0.0,1.904762,1.108614,0.000000,0.000000
83,BOS,2022,1.0,3.0,1.0,2.0,1.0,2.0,0.0,2.0,1.333333,0.471405,1.333333,0.942809
91,BRK,2022,2.0,2.0,1.0,1.0,1.0,0.0,0.0,0.0,1.000000,0.000000,0.000000,0.000000
152,CHI,2022,2.0,2.0,1.0,2.0,2.0,1.0,0.0,2.0,1.523810,0.499433,0.857143,0.709508
157,CHO,2022,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.000000,0.000000,0.476190,0.499433
192,CLE,2022,2.0,0.0,2.0,3.0,1.0,3.0,3.0,3.0,2.142857,0.709508,3.000000,0.000000
220,DAL,2022,1.0,3.0,1.0,1.0,1.0,0.0,0.0,0.0,1.000000,0.000000,0.000000,0.000000
247,DEN,2022,1.0,3.0,1.0,1.0,2.0,3.0,0.0,2.0,1.190476,0.392677,1.809524,1.331632
335,GSW,2022,3.0,2.0,3.0,4.0,4.0,5.0,4.0,5.0,3.523810,0.499433,4.666667,0.471405
495,LAL,2022,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.000000,0.000000,0.476190,0.499433


Il faut realiser la meme chose pour les statistiques team_per_100 et op_team_per_100

In [41]:
# Trier par équipe et saison
team_p100 = team_p100.sort_values(["team", "season"]).copy()
op_team_p100 = op_team_p100.sort_values(["team", "season"]).copy()


# Statistiques Per 100 à utiliser
p100_cols = p100_cols = [
    "fg_per_100_poss",
    "fga_per_100_poss",
    "fg_percent",
    "x3p_per_100_poss",
    "x3pa_per_100_poss",
    "x3p_percent",
    "x2p_per_100_poss",
    "x2pa_per_100_poss",
    "x2p_percent",
    "ft_per_100_poss",
    "fta_per_100_poss",
    "ft_percent",
    "orb_per_100_poss",
    "drb_per_100_poss",
    "trb_per_100_poss",
    "ast_per_100_poss",
    "stl_per_100_poss",
    "blk_per_100_poss",
    "tov_per_100_poss",
    "pf_per_100_poss",
    "pts_per_100_poss"
]

op_team_p100_cols = [
 'opp_fg_per_100_poss',
 'opp_fga_per_100_poss',
 'opp_fg_percent',
 'opp_x3p_per_100_poss',
 'opp_x3pa_per_100_poss',
 'opp_x3p_percent',
 'opp_x2p_per_100_poss',
 'opp_x2pa_per_100_poss',
 'opp_x2p_percent',
 'opp_ft_per_100_poss',
 'opp_fta_per_100_poss',
 'opp_ft_percent',
 'opp_orb_per_100_poss',
 'opp_drb_per_100_poss',
 'opp_trb_per_100_poss',
 'opp_ast_per_100_poss',
 'opp_stl_per_100_poss',
 'opp_blk_per_100_poss',
 'opp_tov_per_100_poss',
 'opp_pf_per_100_poss',
 'opp_pts_per_100_poss']

# TEAM PER 100
for col in p100_cols:

    team_p100[f"{col}_prev1"] = (
        team_p100.groupby("abbreviation")[col].shift(1)
    )

    team_p100[f"{col}_prev2"] = (
        team_p100.groupby("abbreviation")[col].shift(2)
    )

    team_p100[f"{col}_prev3"] = (
        team_p100.groupby("abbreviation")[col].shift(3)
    )

    prev_cols = [
        f"{col}_prev1",
        f"{col}_prev2",
        f"{col}_prev3"
    ]

    team_p100[[f"{col}_mean", f"{col}_std"]] = (
        team_p100[prev_cols].apply(
            lambda row: weighted_mean_std(
                row,
                prev_cols,
                weights
            ),
            axis=1
        )
    )


# OPPONENT PER 100
for col in op_team_p100_cols:

    op_team_p100[f"{col}_prev1"] = (
        op_team_p100.groupby("abbreviation")[col].shift(1)
    )

    op_team_p100[f"{col}_prev2"] = (
        op_team_p100.groupby("abbreviation")[col].shift(2)
    )

    op_team_p100[f"{col}_prev3"] = (
        op_team_p100.groupby("abbreviation")[col].shift(3)
    )

    prev_cols = [
        f"{col}_prev1",
        f"{col}_prev2",
        f"{col}_prev3"
    ]

    op_team_p100[[f"{col}_mean", f"{col}_std"]] = (
        op_team_p100[prev_cols].apply(
            lambda row: weighted_mean_std(
                row,
                prev_cols,
                weights
            ),
            axis=1
        )
    )

C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\4006305716.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  op_team_p100[[f"{col}_mean", f"{col}_std"]] = (
C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\4006305716.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  op_team_p100[[f"{col}_mean", f"{col}_std"]] = (


In [42]:
display(
    team_p100[
        (team_p100["abbreviation"] == "MIL") &
        (team_p100["season"] == 2020)
    ][[
        "team",
        "season",
        "pts_per_100_poss",
        "pts_per_100_poss_prev1",
        "pts_per_100_poss_prev2",
        "pts_per_100_poss_prev3",
        "pts_per_100_poss_mean",
        "pts_per_100_poss_std"
    ]]
)

,team,season,pts_per_100_poss,pts_per_100_poss_prev1,pts_per_100_poss_prev2,pts_per_100_poss_prev3,pts_per_100_poss_mean,pts_per_100_poss_std
196,Milwaukee Bucks,2020,112.4,113.8,109.8,109.1,111.571429,2.13879


In [43]:
display(
    op_team_p100[
        (op_team_p100["abbreviation"] == "MIL") &
        (op_team_p100["season"] == 2022)
    ][[
        "team",
        "season",
        "opp_pts_per_100_poss",
        "opp_pts_per_100_poss_prev1",
        "opp_pts_per_100_poss_prev2",
        "opp_pts_per_100_poss_prev3",
        "opp_pts_per_100_poss_mean",
        "opp_pts_per_100_poss_std"
    ]]
)

,team,season,opp_pts_per_100_poss,opp_pts_per_100_poss_prev1,opp_pts_per_100_poss_prev2,opp_pts_per_100_poss_prev3,opp_pts_per_100_poss_mean,opp_pts_per_100_poss_std
136,Milwaukee Bucks,2022,111.8,111.4,102.9,105.2,107.385714,3.910339


In [44]:

team_p100["playoffs"] = team_p100["playoffs"].astype(float)

team_p100["playoffs_prev1"] = (
    team_p100.groupby("abbreviation")["playoffs"].shift(1)
)

team_p100["playoffs_prev2"] = (
    team_p100.groupby("abbreviation")["playoffs"].shift(2)
)

team_p100["playoffs_prev3"] = (
    team_p100.groupby("abbreviation")["playoffs"].shift(3)
)

playoff_cols = [
    "playoffs_prev1",
    "playoffs_prev2",
    "playoffs_prev3"
]

team_p100[["playoffs_mean", "playoffs_std"]] = (
    team_p100[playoff_cols].apply(
        lambda row: weighted_mean_std(
            row,
            playoff_cols,
            weights
        ),
        axis=1
    )
)

C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\1751787191.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_p100[["playoffs_mean", "playoffs_std"]] = (
C:\Users\marsr\AppData\Local\Temp\ipykernel_36256\1751787191.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  team_p100[["playoffs_mean", "playoffs_std"]] = (


In [45]:
display(
    team_p100.loc[
        (team_p100["abbreviation"] == "GSW") &
        (team_p100["season"] >= 2020),
        [
            "season",
            "playoffs",
            "playoffs_prev1",
            "playoffs_prev2",
            "playoffs_prev3",
            "playoffs_mean",
            "playoffs_std"
        ]
    ]
)

,season,playoffs,playoffs_prev1,playoffs_prev2,playoffs_prev3,playoffs_mean,playoffs_std
189,2020,0.0,1.0,1.0,1.0,1.000000,0.000000
159,2021,0.0,0.0,1.0,1.0,0.523810,0.499433
129,2022,1.0,0.0,0.0,1.0,0.190476,0.392677
99,2023,1.0,1.0,0.0,0.0,0.476190,0.499433
69,2024,0.0,1.0,1.0,0.0,0.809524,0.392677
39,2025,1.0,0.0,1.0,1.0,0.523810,0.499433
9,2026,0.0,1.0,0.0,1.0,0.666667,0.471405


On peut desormais creer le dataset final qu on utilisera pour entrainer les modeles avec les donneees voulues

In [46]:
team_awards = team_awards.rename(columns={"team": "abbreviation"})

# Team Per 100
team_p100_features = team_p100[
    ["abbreviation", "season"] +
    [
        col for col in team_p100.columns
        if col.endswith("_mean") or col.endswith("_std")
    ]
].copy()


# Opponent Per 100
op_team_p100_features = op_team_p100[
    ["abbreviation", "season"] +
    [
        col for col in op_team_p100.columns
        if col.endswith("_mean") or col.endswith("_std")
    ]
].copy()


# Awards
team_awards_features = team_awards[
    [
        "abbreviation",
        "season",
        "all_star_mean",
        "all_star_std",
        "all_nba_mean",
        "all_nba_std"
    ]
].copy()

In [47]:
target = team_summary[
    ["abbreviation", "season", "w"]
].copy()

In [48]:
nba_ml_dataset = team_p100_features.merge(
    op_team_p100_features,
    on=["abbreviation", "season"],
    how="left",
    suffixes=("", "_opp")
)

nba_ml_dataset = nba_ml_dataset.merge(
    team_awards_features,
    on=["abbreviation", "season"],
    how="left"
)

nba_ml_dataset = nba_ml_dataset.merge(
    target,
    on=["abbreviation", "season"],
    how="left"
)

# Garder uniquement les observations pour lesquelles on a une cible
nba_ml_dataset = nba_ml_dataset.dropna(subset=["w"]).copy()